> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [Control Plane概要](#control-plane概要)
- [Fleet Overview](#fleet-overview)
- [Assets管理](#assets管理)
- [Complianceとセキュリティ](#complianceとセキュリティ)
- [Quota管理](#quota管理)
- [Admin機能](#admin機能)

## 🎯 学習目標

- Control Planeの役割と重要性の理解
- Fleet Overviewを通じた全体システムのモニタリング
- Assets（エージェント、モデル、ツール）の管理方法
- コンプライアンスとセキュリティ設定の構成
- QuotaとRate Limitingの管理
- プロジェクトとユーザー権限の管理

## ⏱️ 予想所要時間

約10分

## Control Plane概要

### Control Planeがとは?

Control Planeは Microsoft Foundryの 中央 管理 センターで, すべての AI リソースを 統合 管理して モニタリングします.

```
Control Plane = モニタリング + 管理 + セキュリティ + ガバナンス
```

### 主要 機能 領域

```
┌─────────────────────────────────────────┐
│         Control Plane                   │
├─────────────────────────────────────────┤
│ Fleet Overview   │ 全体 システム ダッシュボード   │
│ Assets          │ リソース 管理           │
│ Compliance      │ セキュリティ および ポリシー          │
│ Quota           │ クォータ および 制限        │
│ Admin           │ プロジェクト および 権限      │
└─────────────────────────────────────────┘
```

### なぜ 重要したが?

本番環境 環境で 次のを 保証します:
- 📊 **が視性**: すべての リソースの 状態と パフォーマンス 把握
- 🛡️ **セキュリティ**: 脆弱性と 脅威 早期 発見
- 💰 **コスト 管理**: リソース 事容量と コスト 最適化
- ⚖️ **コンプライアンス**: 規定 遵守 状態 モニタリング
- 🚨 **アラート**: 問題 発生 時 即時 対応

### Portal vs SDK

| 機能 | Portal | SDK (Python) |
|-----|--------|-------------|
| Fleet Overview | ✅ ダッシュボード 提供 | ⚠️ 制限的 取得 |
| Assets 取得 | ✅ 可視化 | ✅ プログラミング 方式 |
| Compliance モニタリング | ✅ 実時間 ダッシュボード | ❌ サポート 中 すること |
| Quota 管理 | ✅ Portalで 管理 | ❌ 取得だけ 可能 |
| Admin 設定 | ✅ RBAC 管理 | ❌ サポート 中 すること |

**💡 推奨 事項:**
- **Portal**: 実時間 モニタリング, セキュリティ/コンプライアンス 確認, Quota 管理
- **SDK**: 自動化, CI/CD ファがプライン, プログラミング 方式 取得

## 環境設定

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを 見つけられるように)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# が前 ノートブックで 保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (他のツールが使用できるように)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    os.environ["PROJECT_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つかりません.")
    print("💡 01-setup.ipynbを 先に実行して環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント: {PROJECT_ENDPOINT}")


## リソース モニタリング

デプロイされた モデルと エージェントを 取得します.

In [ ]:
# Fleet Overview - デプロイされた エージェント 状態 確認
credential = DefaultAzureCredential()
client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

try:
    # デプロイされた エージェント リスト 取得
    agents = list(client.agents.list())
    
    print("🚀 Fleet Overview - Running Agents")
    print("=" * 80)
    
    if not agents:
        print("\n⚠️ デプロイされた エージェントが ありません.")
        print("💡 03-agents.ipynbを 先に実行してエージェントを作成してください.")
    else:
        print(f"\n📊 合計 {len(agents)}個の エージェントが 実行 中です:\n")
        
        for i, agent in enumerate(agents, 1):
            print(f"{i}. {agent.name}")
            print(f"   ID: {agent.id}")
            
            # Model 情報 (属性名が異なる場合があります)
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'N/A')
            print(f"   Model: {model_name}")
            
            if hasattr(agent, 'tools') and agent.tools:
                tools_str = ", ".join([t.get('type', 'unknown') for t in agent.tools])
                print(f"   Tools: {tools_str}")
            
            if hasattr(agent, 'created_at'):
                print(f"   Created: {agent.created_at}")
            
            print(f"   Status: ✅ Active")
            print("-" * 80)
        
        print(f"\n✅ Fleet Overview 取得 完了!")
        
except Exception as e:
    print(f"⚠️ Fleet 情報 取得 失敗: {e}")
    print("\n💡 解決方法:")
    print("   1. Azureに ログインしているか確認 (az login)")
    print("   2. FOUNDRY_NAMEが 正しいか 確認")
    print("   3. プロジェクトに エージェントが 作成されているか 確認")

## Quota 確認

モデル別 クォータと 使用率を 確認します.

**⚠️ 参考**: 詳細な Quota 管理は Azure Portalでだけ 可能です.
- Operate > Quotaで 実時間 使用率 確認
- Quota 増が リクエスト

In [ ]:
# Assets 管理 - モデル, 接続, も旧 確認

print("📦 Assets 管理")
print("=" * 80)

try:
    # 1. 接続(Connections) 取得
    print("\n1️⃣ Connections (モデル 接続):")
    print("-" * 80)
    
    connections = list(client.connections.list())
    
    if not connections:
        print("⚠️ 接続された モデルが ありません.")
        print("💡 Portalで モデルを デプロイしてください: https://ai.azure.com")
    else:
        for conn in connections:
            print(f"\n✅ {conn.name}")
            if hasattr(conn, 'connection_type'):
                print(f"   Type: {conn.connection_type}")
            if hasattr(conn, 'endpoint_url'):
                print(f"   Endpoint: {conn.endpoint_url}")
    
    # 2. エージェント リスト (Assets)
    print("\n\n2️⃣ Agents:")
    print("-" * 80)
    
    agents = list(client.agents.list())
    
    if agents:
        for agent in agents:
            print(f"\n✅ {agent.name}")
            print(f"   ID: {agent.id}")
            
            # Model 情報 (属性名が異なる場合があります)
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'N/A')
            print(f"   Model: {model_name}")
    else:
        print("⚠️ 作成された エージェントが ありません.")
    
    print("\n" + "=" * 80)
    print("✅ Assets 取得 完了!")
    
    # 3. リソース 統計 可視化
    print("\n\n3️⃣ リソース 統計 可視化:")
    print("-" * 80)
    
    # matplotlib インストール
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    
    import matplotlib.pyplot as plt
    
    # 接続 および エージェント できる 可視化
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # 接続 タイプ別 分布
    if connections:
        connection_types = {}
        for conn in connections:
            if hasattr(conn, 'connection_type'):
                conn_type = str(conn.connection_type)
                connection_types[conn_type] = connection_types.get(conn_type, 0) + 1
        
        if connection_types:
            ax1.bar(connection_types.keys(), connection_types.values(), color='skyblue')
            ax1.set_title('Connection Types Distribution', fontsize=14, fontweight='bold')
            ax1.set_xlabel('Connection Type')
            ax1.set_ylabel('Count')
            ax1.tick_params(axis='x', rotation=45)
    else:
        ax1.text(0.5, 0.5, 'No connections available', ha='center', va='center', fontsize=12)
        ax1.set_title('Connection Types Distribution', fontsize=14, fontweight='bold')
    
    # エージェント モデル 分布
    if agents:
        agent_models = {}
        for agent in agents:
            model_name = getattr(agent, 'model', None) or getattr(agent, 'model_id', 'Unknown')
            agent_models[model_name] = agent_models.get(model_name, 0) + 1
        
        ax2.bar(agent_models.keys(), agent_models.values(), color='lightcoral')
        ax2.set_title('Agent Models Distribution', fontsize=14, fontweight='bold')
        ax2.set_xlabel('Model')
        ax2.set_ylabel('Count')
        ax2.tick_params(axis='x', rotation=45)
    else:
        ax2.text(0.5, 0.5, 'No agents available', ha='center', va='center', fontsize=12)
        ax2.set_title('Agent Models Distribution', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\\n📊 リソース 統計 可視化 完了!")
    
except Exception as e:
    print(f"\n⚠️ Assets 取得 失敗: {e}")
    print("💡 Portalで 確認: https://ai.azure.com > Build > Assets")

## リソース 整理 (選択事項)

ワークショップ 終了 後 コスト 削減を ために リソースを 削除する できる あります.

In [ ]:
# Quota 管理 情報
print("📊 Quota および Rate Limiting 情報")
print("=" * 80)
print("\n💡 Quota および Rate Limitingは 主で Azure Portalで 管理なります.")
print("\n確認 方法:")
print("   1. Azure Portal: https://portal.azure.com")
print("   2. AI Services リソース 選択")
print("   3. 'Quota' または 'Usage + quotas' メニュー 確認")
print("\n主要 確認 項目:")
print("   - Token Per Minute (TPM): 分あたり 処理 可能な トークン できる")
print("   - Requests Per Minute (RPM): 分あたり 処理 可能な リクエスト できる")
print("   - Provisioning Throughput Units (PTU): プでビジョニングされた 処理 単位")
print("\n" + "=" * 80)
print("✅ Portalで 実時間 quota 事容量を 確認してください!")

## 📚 追加リソース

- [Microsoft Foundry Control Plane](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/overview?view=foundry)
- [Agent ヘルス および パフォーマンス モニタリング](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/monitoring-across-fleet?view=foundry)
- [Custom Agent 登録](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/register-custom-agent?view=foundry)
- [Guardrail Policy 作成](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/quickstart-create-guardrail-policy?view=foundry)
- [規定 遵守 および セキュリティ 管理](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/how-to-manage-compliance-security?view=foundry)
- [モデル コスト および パフォーマンス 最適化](https://learn.microsoft.com/en-us/azure/ai-foundry/control-plane/how-to-optimize-cost-performance?view=foundry)

## まとめ

### おめでとうします! 🎉

Microsoft Foundry Hands-on Workshopの すべての モジュールを 完了しました!

### 学習した 内容 概要

```
✅ Module 01: 環境設定
   - Resource Group および Foundry リソース 作成

✅ Module 02: モデル および デプロイ
   - モデル 探索, デプロイ, Model Router 構成

✅ Module 03: エージェント 開発
   - 様々な タイプの エージェント 作成 および デプロイ

✅ Module 04: Foundry IQ
   - AI Search および Blob Storage ベースのKnowledge Base 構築

✅ Module 05: ワークフロー
   - Sequential, Group Chat, Human-in-loop ワークフロー 実装

✅ Module 06: 評価
   - エージェント 品質 評価 および 改善

✅ Module 07: Control Plane
   - 本番環境 モニタリング および 管理
```

### 次のステップ

が第 次のを 実行する 準備が なりました:

1. **本番環境 デプロイ**
   - 実際 アプリケーションに Foundry 統合
   - CI/CD ファがプライン 構築
   - モニタリング および アラート 設定

2. **高度な 機能 探索**
   - Custom tools および MCP servers 開発
   - Multi-agent 協業 パターン
   - Fine-tuning および モデル カスタマが徴

3. **コミュニティ 参加**
   - [Microsoft Tech Community](https://techcommunity.microsoft.com) フォーラム 参加
   - [GitHub Samples](https://github.com/Azure-Samples) 貢献
   - 使用 事例 共有